In [1]:
import lightgbm as lgb
import numpy as np
import optuna
import polars as pl
from sklearn.model_selection import train_test_split
from surrogate_model.metrics import enrichment_factor, spearman_corr, top_k_recall
from surrogate_model.optuna import (
    RECALL_TOP_1_PERCENT,
    RECALL_TOP_5_PERCENT,
    make_objective,
)


In [2]:
FEATURES = "data/candidates.10k.parquet"
LABELS = "data/1L83.1L83:p2rank:2.10k.parquet"

RANDOM_SEED = 1000
SAMPLE_SIZE = 10000

In [3]:
features = pl.read_parquet(FEATURES)

features = features.filter(
    (pl.col("parse_ok"))
    & (pl.col("error") == "SUCCESS")
    & (pl.col("conversion_error") == "SUCCESS")
)

cns_mpo_schema = pl.Struct(
    [
        pl.Field("clogp", pl.Float64),
        pl.Field("clogd", pl.Float64),
        pl.Field("tpsa", pl.Float64),
    ]
)

features = features.with_columns(
    pl.col("cns_mpo_components").str.json_decode(cns_mpo_schema)
).unnest("cns_mpo_components")

In [4]:
features = features.drop(
    ["smiles", "parse_ok", "pains_flags", "error", "conversion_error"]
)

In [5]:
labels = pl.read_parquet(LABELS)
labels = labels["catalog_id", "affinity_kcal_mol"]

In [6]:
df = features.join(labels, on="catalog_id", how="inner")

In [7]:
if SAMPLE_SIZE < len(df):
    df = df.sample(SAMPLE_SIZE, seed=RANDOM_SEED)

In [8]:
FEATURE_NAMES = ["heavy_atom_count", "molecular_weight", "clogp", "clogd", "tpsa"]

LABEL_NAME = "affinity_kcal_mol"

In [9]:
x = df.select(FEATURE_NAMES).to_numpy()
x = np.hstack([x, np.array(df["morgan_fp"].to_list())])

In [10]:
y = df[LABEL_NAME].to_numpy()

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=RANDOM_SEED
)

In [12]:
study = optuna.create_study(direction="minimize")

[I 2026-08-14 23:42:49,878] A new study created in memory with name: no-name-35a75316-96b8-4a52-b99a-8f4a50a1adf4


In [13]:
study.optimize(
    make_objective(
        X_train,
        y_train,
        5,
        primary_metric=RECALL_TOP_5_PERCENT,
        random_seed=RANDOM_SEED,
    ),
    n_trials=20,
    show_progress_bar=True,
)

  0%|          | 0/20 [00:00<?, ?it/s]

[I 2026-08-14 23:42:53,719] Trial 0 finished with value: 0.3657894736842105 and parameters: {'num_leaves': 160, 'max_depth': 7, 'learning_rate': 0.15600431201048504, 'n_estimators': 676, 'min_child_samples': 6, 'subsample': 0.994048244484683, 'colsample_bytree': 0.5424295491654443, 'reg_alpha': 0.00012992315286649824, 'reg_lambda': 0.00010158129514273075}. Best is trial 0 with value: 0.3657894736842105.
[I 2026-08-14 23:42:54,648] Trial 1 finished with value: 0.3973684210526316 and parameters: {'num_leaves': 212, 'max_depth': 3, 'learning_rate': 0.01910717469987937, 'n_estimators': 511, 'min_child_samples': 80, 'subsample': 0.9656047833287779, 'colsample_bytree': 0.7735765434002992, 'reg_alpha': 0.053364112163629546, 'reg_lambda': 2.1918482708893516e-05}. Best is trial 0 with value: 0.3657894736842105.
[I 2026-08-14 23:42:59,681] Trial 2 finished with value: 0.38421052631578945 and parameters: {'num_leaves': 139, 'max_depth': 6, 'learning_rate': 0.0012824048298997115, 'n_estimators': 1

In [14]:
best_params = study.best_params
best_params.update({"random_state": RANDOM_SEED})

final_model = lgb.LGBMRegressor(**best_params)
final_model.fit(
    X_train,
    y_train,
    eval_X=X_test,
    eval_y=y_test,
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=True)],
)

Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[118]	valid_0's l2: 215.774


,num_leaves,222
,max_depth,3
,learning_rate,0.005383426895186431
,n_estimators,118
,min_child_samples,49
,subsample,0.879394583721494
,colsample_bytree,0.9089378286785782
,reg_alpha,2.201872166464377e-05
,reg_lambda,3.331875845963912
,random_state,1000
,boosting_type,'gbdt'


In [15]:
y_pred = final_model.predict(X_test)

In [16]:
results = {
    "top_1_percent": top_k_recall(y_test, y_pred, 0.01),
    "top_5_percent": top_k_recall(y_test, y_pred, 0.05),
    "top_10_percent": top_k_recall(y_test, y_pred, 0.1),
    "spearman": spearman_corr(y_test, y_pred),
    "enrichment_factor": enrichment_factor(y_test, y_pred, 0.05),
}

In [17]:
results

{'top_1_percent': 0.3,
 'top_5_percent': 0.3020833333333333,
 'top_10_percent': 0.45549738219895286,
 'spearman': np.float64(0.8440132062330298),
 'enrichment_factor': 6.041666666666666}

# Results

| dataset size | trials | top_1_percent |top_5_percent| top_10_percent | spearman | enrichment |
| - | - | - | - | - | - | - |
| 10k - top 1 | 20 | 0.2 | 0.406 | 0.513 | 0.83 | 8.125 |
| 10k - top 5 | 20 | 0.3 | 0.30 | 0.455 | 0.84 | 6.04 |
